In [1]:
!pip -q install gradio pandas numpy scikit-learn

import pandas as pd
import numpy as np
import gradio as gr
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

from google.colab import drive

drive.mount('/content/drive')

path = "/content/drive/MyDrive/Dataset/Huntington_Disease_Dataset.csv"
df = pd.read_csv(path)

DATASET_COLS = [
    "Patient_ID",
    "Age","Sex","Family_History","HTT_CAG_Repeat_Length",
    "Motor_Symptoms","Cognitive_Decline","Chorea_Score",
    "Brain_Volume_Loss","Functional_Capacity","Gene_Mutation_Type","Category"
]
DATASET_COLS = [c for c in DATASET_COLS if c in df.columns]

# Scales
DIFFICULTY_5_OPTIONS = [
    "No: I have no difficulty doing this.",
    "Mild: I have difficulty doing this but can still do it well.",
    "Moderate: I have difficulty doing this a lot.",
    "Severe: I can no longer do this.",
    "I never did this activity."
]
DIFF_SCORE = {
    DIFFICULTY_5_OPTIONS[0]: 0,
    DIFFICULTY_5_OPTIONS[1]: 1,
    DIFFICULTY_5_OPTIONS[2]: 2,
    DIFFICULTY_5_OPTIONS[3]: 3,
    DIFFICULTY_5_OPTIONS[4]: 0,
}

LIKERT_1_7 = ["1","2","3","4","5","6","7"]
STRESS_1_5 = ["1","2","3","4","5"]

AGREE_7_HEADER = (
    "Please continue to think about your life now.\n\n"
    "Below you are provided with a list of statements.\n"
    "Please read each statement carefully and indicate the extent to which you agree or disagree.\n\n"
    "1 = Strongly disagree   ...   7 = Strongly agree\n"
    "Tip: If you are unsure, choose 4 (Neutral)."
)

OPTIMISM_7_HEADER = (
    "In general, how optimistic would you say you are?\n\n"
    "1 = Not at all   ...   7 = Very much\n"
    "Tip: If you are unsure, choose 4 (Average)."
)

STRESS_5_HEADER = (
    "The following items are about stress.\n"
    "Please tell us how much each statement causes you to feel stress.\n\n"
    "1 = No stress   ...   5 = A lot of stress"
)

INTRO_TEXT = (
    "Hi! 👋 I’m the Huntington’s Disease Survey Chatbot.\n\n"
    "I’ll ask you questions in groups (like a survey). Click one option (bubble) and press Send.\n\n"
    "Important: This tool is for screening and education only. It does NOT confirm a medical diagnosis.\n"
)

# Similarity matching (hidden from user wording; used only for summary table)
def find_similar(user_row):
    cols_for_match = [c for c in DATASET_COLS if c not in ["Patient_ID"]]
    if not cols_for_match:
        return None

    base = df[cols_for_match].copy()
    user = pd.DataFrame([user_row])[cols_for_match]

    combined = pd.concat([base, user], ignore_index=True)
    combined = pd.get_dummies(combined, dummy_na=True).fillna(0)

    scaled = StandardScaler(with_mean=False).fit_transform(combined)
    sims = cosine_similarity(scaled[-1:], scaled[:-1])[0]
    top = np.argsort(sims)[::-1][:5]

    out = df.iloc[top].copy()
    out["_similarity"] = sims[top]
    return out

# Question list
QUESTIONS = []

# Eligibility
QUESTIONS += [
    {"group":"Eligibility","key":"age18","type":"radio",
     "q":"Are you 18 years of age or older?",
     "opt":["Yes","No"]},

    {"group":"Eligibility","key":"english_understand","type":"radio",
     "q":"Are you able to speak and understand English?",
     "opt":["Yes","No"]},
]

# Demographics (no email)
QUESTIONS += [
    {"group":"Survey 1 of 3: Demographics","key":"english_primary_secondary","type":"radio",
     "q":"Is English your primary or secondary language?",
     "opt":["Primary","Secondary"]},

    {"group":"Survey 1 of 3: Demographics","key":"gender","type":"radio",
     "q":"What is your gender?",
     "opt":["Male","Female","Prefer not to say"]},

    {"group":"Survey 1 of 3: Demographics","key":"country","type":"radio",
     "q":"Which country do you live in?",
     "opt":["Pakistan","India","United States","United Kingdom","Canada","Other / Prefer not to say"]},

    {"group":"Survey 1 of 3: Demographics","key":"diagnosed_based_on","type":"radio",
     "q":"Have you ever been diagnosed with Huntington’s disease?",
     "opt":["No diagnosis","Genetic test","Motor symptoms","Both","I prefer not to say"]},
]

# Daily life (difficulty)
QUESTIONS += [
    {"group":"Survey 2 of 3: Daily Life (Past two weeks)","key":"diff_speech","type":"radio",
     "q":"Over the past two weeks, did you have difficulty with your speech?\n\n"
        "Examples: slurring, trouble finding words, or people not understanding you.",
     "opt":DIFFICULTY_5_OPTIONS, "score":"diff"},

    {"group":"Survey 2 of 3: Daily Life (Past two weeks)","key":"diff_interacting","type":"radio",
     "q":"Over the past two weeks, did you have difficulty interacting with friends, relatives, or colleagues?\n\n"
        "Examples: keeping up with conversation, staying in touch, or discussing things at work.",
     "opt":DIFFICULTY_5_OPTIONS, "score":"diff"},

    {"group":"Survey 2 of 3: Daily Life (Past two weeks)","key":"diff_planning_day","type":"radio",
     "q":"Over the past two weeks, did you have difficulty planning how to spend your day?\n\n"
        "Examples: deciding what to do, planning appointments, meals, or meeting people.",
     "opt":DIFFICULTY_5_OPTIONS, "score":"diff"},

    {"group":"Survey 2 of 3: Daily Life (Past two weeks)","key":"diff_hands","type":"radio",
     "q":"Over the past two weeks, did you have difficulty using your hands?\n\n"
        "Examples: writing, buttons, keys, holding utensils, or picking up small objects.",
     "opt":DIFFICULTY_5_OPTIONS, "score":"diff"},

    {"group":"Survey 2 of 3: Daily Life (Past two weeks)","key":"diff_walking","type":"radio",
     "q":"Over the past two weeks, did you have difficulty walking?\n\n"
        "Examples: balance problems, stairs, feeling unsteady, or walking smoothly.",
     "opt":DIFFICULTY_5_OPTIONS, "score":"diff"},
]

# Thinking & coping (like screenshot style)
QUESTIONS += [
    {"group":"Thinking & Coping","key":"agree_header","type":"radio",
     "q":AGREE_7_HEADER, "opt":["Continue"]},

    {"group":"Thinking & Coping","key":"loc_1","type":"radio",
     "q":"I am not in control of most things that occur in my life.",
     "opt":LIKERT_1_7},

    {"group":"Thinking & Coping","key":"loc_2","type":"radio",
     "q":"The events in my life are mainly determined by my own actions.",
     "opt":LIKERT_1_7},

    {"group":"Thinking & Coping","key":"loc_3","type":"radio",
     "q":"What happens in my life is often beyond my control.",
     "opt":LIKERT_1_7},

    {"group":"Thinking & Coping","key":"loc_4","type":"radio",
     "q":"Whether or not I am able to get what I want is in my own hands.",
     "opt":LIKERT_1_7},
]

# Optimism
QUESTIONS += [
    {"group":"Optimism","key":"optimism_header","type":"radio",
     "q":OPTIMISM_7_HEADER, "opt":["Continue"]},

    {"group":"Optimism","key":"optimism","type":"radio",
     "q":"Please select one option.",
     "opt":LIKERT_1_7},
]

# Stress
QUESTIONS += [
    {"group":"Stress & Worries","key":"stress_header","type":"radio",
     "q":STRESS_5_HEADER, "opt":["Continue"]},

    {"group":"Stress & Worries","key":"stress_1","type":"radio",
     "q":"Worrying that I will develop Huntington’s disease.",
     "opt":STRESS_1_5},

    {"group":"Stress & Worries","key":"stress_2","type":"radio",
     "q":"Feeling as though no one knows what I am going through.",
     "opt":STRESS_1_5},

    {"group":"Stress & Worries","key":"stress_3","type":"radio",
     "q":"Feeling judged because other people don’t understand Huntington’s disease.",
     "opt":STRESS_1_5},
]

# Health & symptom questions (these replace the confusing "dataset" wording)
HEALTH_DETAILS = []

if "Age" in df.columns:
    HEALTH_DETAILS.append({
        "group":"Health & Thinking Abilities",
        "key":"ds_Age",
        "type":"radio",
        "q":"How old are you?",
        "opt":["18–25","26–35","36–45","46–55","56–65","66+","I don’t know"],
        "dataset":"Age",
        "convert": lambda x: {"18–25":22,"26–35":30,"36–45":40,"46–55":50,"56–65":60,"66+":70,"I don’t know":np.nan}.get(x, np.nan)
    })

if "Sex" in df.columns:
    HEALTH_DETAILS.append({
        "group":"Health & Thinking Abilities",
        "key":"ds_Sex",
        "type":"radio",
        "q":"What is your sex?",
        "opt":["Male","Female","Prefer not to say"],
        "dataset":"Sex",
        "convert": lambda x: x
    })

if "Family_History" in df.columns:
    HEALTH_DETAILS.append({
        "group":"Health & Thinking Abilities",
        "key":"ds_Family_History",
        "type":"radio",
        "q":"Has anyone in your close family (parent or sibling) had Huntington’s disease?",
        "opt":["Yes","No","Not sure"],
        "dataset":"Family_History",
        "convert": lambda x: {"Yes":1,"No":0,"Not sure":np.nan}.get(x, np.nan)
    })

if "Motor_Symptoms" in df.columns:
    HEALTH_DETAILS.append({
        "group":"Movement & Balance",
        "key":"ds_Motor_Symptoms",
        "type":"radio",
        "q":"Do you currently notice movement problems?\n\nExamples: jerky movements (chorea), balance issues, stiffness, or clumsiness.",
        "opt":["No","Mild","Moderate","Severe","Not sure"],
        "dataset":"Motor_Symptoms",
        "convert": lambda x: {"No":0,"Mild":1,"Moderate":2,"Severe":3,"Not sure":np.nan}.get(x, np.nan)
    })

if "Cognitive_Decline" in df.columns:
    HEALTH_DETAILS.append({
        "group":"Health & Thinking Abilities",
        "key":"ds_Cognitive_Decline",
        "type":"radio",
        "q":"Do you notice thinking or memory problems?\n\nExamples: forgetting, slower thinking, trouble focusing, or difficulty planning.",
        "opt":["No","Mild","Moderate","Severe","Not sure"],
        "dataset":"Cognitive_Decline",
        "convert": lambda x: {"No":0,"Mild":1,"Moderate":2,"Severe":3,"Not sure":np.nan}.get(x, np.nan)
    })

if "Functional_Capacity" in df.columns:
    HEALTH_DETAILS.append({
        "group":"Daily Independence",
        "key":"ds_Functional_Capacity",
        "type":"radio",
        "q":"How independent are you in daily life?\n\nThink about work, finances, household tasks, and personal care.",
        "opt":[
            "Fully independent",
            "Mostly independent (needs small help sometimes)",
            "Needs regular help",
            "Needs full-time help",
            "Not sure"
        ],
        "dataset":"Functional_Capacity",
        "convert": lambda x: {
            "Fully independent":3,
            "Mostly independent (needs small help sometimes)":2,
            "Needs regular help":1,
            "Needs full-time help":0,
            "Not sure":np.nan
        }.get(x, np.nan)
    })

if "Gene_Mutation_Type" in df.columns:
    uniq = sorted(df["Gene_Mutation_Type"].dropna().astype(str).unique().tolist())
    HEALTH_DETAILS.append({
        "group":"Genetic Information",
        "key":"ds_Gene_Mutation_Type",
        "type":"radio",
        "q":"If you have genetic test results, which type of mutation was reported?",
        "opt":["I don’t know"] + uniq,
        "dataset":"Gene_Mutation_Type",
        "convert": lambda x: np.nan if x == "I don’t know" else x
    })

if "Category" in df.columns:
    uniq = sorted(df["Category"].dropna().astype(str).unique().tolist())
    HEALTH_DETAILS.append({
        "group":"Genetic Information",
        "key":"ds_Category",
        "type":"radio",
        "q":"If you have a report, which category best matches it?",
        "opt":["I don’t know"] + uniq,
        "dataset":"Category",
        "convert": lambda x: np.nan if x == "I don’t know" else x
    })

QUESTIONS += HEALTH_DETAILS

def init_state():
    return {"i": 0, "ans": {}}

def severity_label(diff_total):
    if diff_total <= 6:
        return "Low difficulty (mild)"
    if diff_total <= 15:
        return "Moderate difficulty"
    return "High difficulty (severe)"

def final_suggestions(diff_total, family_hist, motor_level, cog_level):
    tips = []
    if diff_total >= 16 or motor_level in [2,3] or cog_level in [2,3]:
        tips += [
            "• If symptoms are affecting daily life, consider seeing a neurologist.",
            "• If walking/balance feels unsafe, prioritize safety (support, avoid risky situations).",
            "• Consider support for stress, anxiety, or depression if present.",
        ]
    elif diff_total >= 7:
        tips += [
            "• Track symptoms for 2–4 weeks and share notes with a doctor.",
            "• Prioritize sleep, routine, and stress reduction.",
        ]
    else:
        tips += [
            "• If you are worried due to family history, consider genetic counseling.",
            "• If symptoms develop later, repeat this survey or consult a clinician.",
        ]

    if family_hist == 1:
        tips.append("• Because you reported family history, genetic counseling may be especially helpful.")

    return "Suggestions:\n" + "\n".join(tips)

def build_prompt(q):
    return f"Group: {q['group']}\n\n{q['q']}"

def show_input(q):
    if q["type"] == "radio":
        return (
            gr.update(visible=True, choices=q["opt"], value=None, label="Select one"),
            gr.update(visible=False, value="")
        )
    return (
        gr.update(visible=False, choices=[], value=None),
        gr.update(visible=True, value="", label="Type your answer")
    )

def start():
    s = init_state()
    history = [("", INTRO_TEXT + "\n\n" + build_prompt(QUESTIONS[0]))]
    r, t = show_input(QUESTIONS[0])
    return history, s, r, t

def format_matches_table(matches):
    if matches is None or len(matches) == 0:
        return "No similar records found."

    cols = []
    if "Patient_ID" in matches.columns:
        cols.append("Patient_ID")
    for c in ["Age","Sex","Family_History","Motor_Symptoms","Cognitive_Decline","Functional_Capacity","Category"]:
        if c in matches.columns:
            cols.append(c)
    cols.append("_similarity")

    view = matches[cols].copy()
    view["_similarity"] = (view["_similarity"] * 100).round(1).astype(str) + "%"

    return view.to_string(index=False)

def send_answer(state, history, r_val, t_val):
    q = QUESTIONS[state["i"]]

    if q["type"] == "radio":
        answer = r_val
        if answer is None:
            history.append(("", "Please select one option (click a bubble), then press Send."))
            return history, state, *show_input(q)
    else:
        answer = (t_val or "").strip()
        if answer == "":
            history.append(("", "Please type an answer."))
            return history, state, *show_input(q)

    history.append((str(answer), ""))

    state["ans"][q["key"]] = answer
    state["i"] += 1

    if state["i"] >= len(QUESTIONS):
        diff_total = 0
        for qq in QUESTIONS:
            if qq.get("score") == "diff":
                diff_total += DIFF_SCORE.get(state["ans"].get(qq["key"]), 0)

        user_row = {c: np.nan for c in DATASET_COLS if c != "Patient_ID"}
        for qq in QUESTIONS:
            if qq.get("dataset"):
                col = qq["dataset"]
                val = state["ans"].get(qq["key"], np.nan)
                conv = qq.get("convert")
                if conv is not None:
                    try:
                        val = conv(val)
                    except Exception:
                        val = np.nan
                user_row[col] = val

        matches = find_similar(user_row)

        family_hist = user_row.get("Family_History", np.nan)
        motor_level = user_row.get("Motor_Symptoms", np.nan)
        cog_level = user_row.get("Cognitive_Decline", np.nan)
        if not np.isfinite(motor_level): motor_level = -1
        if not np.isfinite(cog_level): cog_level = -1

        summary = (
            "✅ Survey completed.\n\n"
            f"Daily-life difficulty severity: **{severity_label(diff_total)}**\n\n"
            "Important note:\n"
            "This is a screening tool only and cannot confirm Huntington’s disease.\n\n"
            f"{final_suggestions(diff_total, family_hist, motor_level, cog_level)}\n\n"
            "People in the dataset with similar answers:\n"
            f"{format_matches_table(matches)}"
        )

        history.append(("", summary))
        return history, state, gr.update(visible=False), gr.update(visible=False)

    next_q = QUESTIONS[state["i"]]
    history.append(("", build_prompt(next_q)))
    return history, state, *show_input(next_q)

with gr.Blocks() as demo:
    gr.Markdown("# Huntington’s Disease Survey Chatbot")

    chat = gr.Chatbot(height=560)
    state = gr.State(init_state())

    radio = gr.Radio(choices=[], visible=False)
    textbox = gr.Textbox(visible=False)

    send = gr.Button("Send")
    restart = gr.Button("Restart")

    demo.load(start, None, [chat, state, radio, textbox])
    send.click(send_answer, inputs=[state, chat, radio, textbox], outputs=[chat, state, radio, textbox])
    restart.click(start, None, [chat, state, radio, textbox])

demo.launch(share=True)


Mounted at /content/drive


/tmp/ipython-input-967153012.py:444: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat = gr.Chatbot(height=560)
/tmp/ipython-input-967153012.py:444: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat = gr.Chatbot(height=560)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3d2732af8c73ac247d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
